# Data exploration

## Events

In [0]:
from pyspark.sql.protobuf.functions import to_protobuf, from_protobuf
from pyspark.sql.types import *
from pyspark.sql import functions as F
import subprocess
import sys

pipeline_path = "/Workspace/Users/mikeias.oliveira@al.infnet.edu.br/dlt-pipeline"
if pipeline_path not in sys.path:
    sys.path.append(pipeline_path)

In [0]:
%skip
%pip install "grpcio-tools>=1.60.0,<2.0"
%restart_python 

In [0]:
import os
from grpc_tools import protoc

# 1. Save your current directory so you can change back later
original_dir = os.getcwd()

try:
    os.chdir("/Volumes/dlt_lakehouse/default/raw/events")

    exit_code = protoc.main(
        [
            "grpc_tools.protoc",
            "-I.",
            "--include_imports",
            f"--descriptor_set_out={pipeline_path}/events.desc",
            "events.proto",
        ]
    )
    print(f"Compilation finished with exit code: {exit_code}")

finally:
    os.chdir(original_dir)

In [0]:
data_file = "/Volumes/dlt_lakehouse/default/raw/events/events_000005.pb"
descriptor_file = "/Workspace/Users/mikeias.oliveira@al.infnet.edu.br/dlt-pipeline/events.desc"

raw_df = spark.read.format("binaryFile").load(data_file)


trimmed_df = raw_df.withColumn("trimmed_content", F.expr("substring(content, 3, length(content) - 2)"))

decoded_df = trimmed_df.withColumn(
    "proto_data",
    from_protobuf(
        data="trimmed_content",
        messageName="factory.smart_manufacturing.Events",
        descFilePath=descriptor_file,
        options={"mode": "PERMISSIVE"}
    ),
)

display(decoded_df)